**Auteur(s)** : Cheikhou Akhmed KANE

**Description** : Collecte de donnees sols pour le fichier `typeDeSolParZH.shp`

## 1. Description du projet

Ce notebook documente le processus de caractérisation des propriétés physiques, hydriques et chimiques des sols pour une zone d'étude définie. La méthodologie repose sur la classification des parcelles en îlots-types homogènes, suivie de l'extraction de données à partir de plusieurs sources géospatiales publiques. Les valeurs moyennes de chaque propriété sont ensuite calculées pour définir un profil caractéristique pour chaque type d'îlot.

---

## 2. Objectifs

Les principaux objectifs de cette étude sont :

* **Extraire** les propriétés physiques (texture, densité), hydriques (rétention en eau) et chimiques (pH, nutriments) du sol pour chaque type d'îlot.
* **Caractériser** chaque îlot-type en calculant les valeurs moyennes des propriétés extraites.
* **Identifier** les données manquantes et définir les prochaines étapes pour finaliser l'analyse.

---

## 3. Méthodologie

### 3.1. Stratégie d'échantillonnage

L'extraction des données a été réalisée en utilisant les **centroïdes géographiques** de chaque parcelle (ainsi que des zones d'influence des arbres -Faidherbia Albida) comme points d'échantillonnage. Les valeurs extraites pour l'ensemble des parcelles d'un même type ont ensuite été agrégées par calcul de la **moyenne**, considérée comme la valeur représentative de l'îlot-type.

### 3.2. Sources de données et variables extraites

#### **OpenLandMap (Soil-DB)**
* **Résolution spatiale :** 120 mètres sauf pour la densite apparente et la teneur en carbone (30m).
* **Propriétés extraites :**
    * Teneur en argile (%).
    * Teneur en sable (%).
    * Densité apparente (kg/m3).
    * Teneur en carbone organique (g/kg), utilisée pour estimer la **teneur en matière organique** via l'application du facteur Van Bemmelen de conversion de 1,724.

#### **iSDA Africa (API)**
* **Résolution spatiale :** 30 mètres.
* **Propriétés extraites :**
    * Teneur en azote total (pour le calcul du rapport C/N).
    * pH du sol.

#### **Cirad Dataverse (J. Lavarenne & L. Leroux, 2024)**
* **Résolution spatiale :** 30 mètres.
* **Propriétés extraites :**
    * Humidité volumique à la capacité au champ.
    * Humidité volumique au point de flétrissement.
    * Réserve utile.

### 3.3. Points de vigilance et données manquantes

* **Hétérogénéité des profondeurs :** Il est à noter que les horizons de sol (profondeurs) pour lesquels les données sont fournies diffèrent selon les sources.
* **Données à compléter :**
    * **Conductivité hydraulique à saturation (KSAT) :** Non disponible directement.
    * **Structure du sol :** En attente des données issues du dire d'experts (CSTRU).

### 3.4. Fonctions Utilitaires

Pour automatiser le traitement, deux fonctions Python principales ont été créées. Ces fonctions permettent de standardiser la lecture des données et l'extraction des valeurs pour assurer la reproductibilité de l'analyse.

* `load_geodata_from_wkt` : Cette fonction charge les données des parcelles depuis un fichier CSV, nettoie les entrées invalides et les convertit en une couche géospatiale (GeoDataFrame) directement utilisable.

* `extract_values` : Cette fonction prend la couche de points et un lien vers une carte raster (un fichier `.tif`). Pour chaque point, elle extrait la valeur correspondante de la carte, en spécifiant au besoin la bande (profondeur) à analyser.

* `get_descriptive_stats` : Prend une colonne numérique du jeu de données et calcule un résumé statistique complet (moyenne, écart-type, min/max, quartiles) pour chaque `type_ilot` défini.

In [4]:
import pandas as pd
import numpy as np
import geopandas as gpd
from shapely.geometry import Point
from pathlib import Path
import rasterio
from rasterio.windows import Window
import matplotlib.pyplot as plt
import seaborn as sns
import gzip
import time
import requests

In [5]:
# 1. Définir le chemin de base comme fourni
base_dir = Path.cwd().parent.resolve()

# 2. Chemins vers les fichiers (entrée et sortie)
input_csv_path = base_dir / "data" / "sols" / "csv" / "processed" / "points_echantillonnage_complets.csv"
output_csv_path = base_dir / "data" / "sols" / "csv" / "processed" / "donnees_typesDeSol.csv"

SOURCE_CRS = "epsg:32628"

## Fonctions utilitaires

In [8]:
def load_geodata_from_wkt(file_path, geometry_col, source_crs):
    """
    Charge un CSV, supprime les lignes avec des géométries nulles,
    et le convertit en GeoDataFrame en utilisant une colonne WKT.
    """
    print(f"Chargement des données depuis : {file_path}")
    if not file_path.exists():
        raise FileNotFoundError(f"ERREUR : Le fichier {file_path} n'a pas été trouvé.")

    df = pd.read_csv(file_path)

    # --- ÉTAPE DE NETTOYAGE ---
    initial_rows = len(df)
    df.dropna(subset=[geometry_col], inplace=True)
    final_rows = len(df)

    if final_rows < initial_rows:
        print(f"Nettoyage : {initial_rows - final_rows} ligne(s) avec des géométries nulles ont été supprimées.")

    # --- Conversion en GeoDataFrame depuis la colonne 'geometry' ---
    gdf = gpd.GeoDataFrame(
        df,
        geometry=gpd.GeoSeries.from_wkt(df[geometry_col]),
        crs=source_crs
    )
    return gdf

In [21]:
# --- Exécution de la fonction ---

try:
    gdf_points = load_geodata_from_wkt(
        file_path=input_csv_path,
        geometry_col='geometry', # La colonne contenant le texte WKT
        source_crs=SOURCE_CRS
    )
    
    print("\n✅ GeoDataFrame chargé avec succès via la fonction.")
    
    # Afficher les dimensions
    print(f"   -> Contient {gdf_points.shape[0]} points (lignes) et {gdf_points.shape[1]} attributs (colonnes).")
    
    display(gdf_points.head())

except FileNotFoundError as e:
    print(e)
except Exception as e:
    print(f"Une autre erreur est survenue : {e}")

Chargement des données depuis : C:\Users\Cheikhou\Desktop\Ferlo_Sine\maelia-data-diohine-v1\data\sols\csv\processed\points_echantillonnage_complets.csv

✅ GeoDataFrame chargé avec succès via la fonction.
   -> Contient 749 points (lignes) et 12 attributs (colonnes).


,parcel_id,N°_PARCEL,TYP_SOL,Arbre,Long,Lat,X_Centroid,Y_Centroid,geometry,Type_champ,type_ilot,ZONE_PEDO
0,b7698961-a77b-4fb2-8c94-2f090c1d6fd2,147.0,Dior,1,337382.976620,1.603607e+06,337365.773977,1.603611e+06,POINT (337382.977 1603607.337),CC,dior_cc_avec_arbr,dior_cc_avec_arbr
1,dae3c8e5-9f33-4354-826d-aed699b6c17c,149.0,Dekk,1,336145.755091,1.603489e+06,336099.653300,1.603497e+06,POINT (336145.755 1603488.988),CB,dekk_cb_avec_arbr,dekk_cb_avec_arbr
2,e85b0517-6fda-4422-95a3-571360932f23,150.0,Dior,1,336296.074335,1.603034e+06,336302.866304,1.603039e+06,POINT (336296.074 1603033.949),CB,dior_cb_avec_arbr,dior_cb_avec_arbr
3,d28eb822-617a-4498-acbf-4f919ee1e3ae,144.0,Dior,1,335917.410954,1.602738e+06,335886.039020,1.602729e+06,POINT (335917.411 1602738.353),CB,dior_cb_avec_arbr,dior_cb_avec_arbr
4,2a2005cb-29a4-48bd-b7ad-fca2bc44db44,148.0,Dior,1,336894.131809,1.603453e+06,336894.334621,1.603456e+06,POINT (336894.132 1603453.182),CB,dior_cb_avec_arbr,dior_cb_avec_arbr


In [10]:
def extract_values(raster_url, gdf, band_number=1):
    """
    Extrait les valeurs d'une bande spécifique d'un raster 
    pour chaque point d'un GeoDataFrame.
    """
    print(f"Extraction des valeurs depuis : {raster_url} (Bande: {band_number})")
    
    with rasterio.open(raster_url) as src:
        gdf_proj = gdf.to_crs(src.crs)

        values = []
        for pt in gdf_proj.geometry:
            try:
                row, col = src.index(pt.x, pt.y)
                window = Window(col, row, 1, 1)
                
                # Utilise le paramètre band_number pour lire la bonne bande
                data = src.read(band_number, window=window)
                
                values.append(data[0, 0])
            except IndexError:
                values.append(None) # Gère le cas où le point est hors du raster

    return values

# Alternative plus rapide (moins lisible)
def extract_values_vectorized(raster_url, gdf, band_number=1):
    with rasterio.open(raster_url) as src:
        gdf_proj = gdf.to_crs(src.crs)
        coords = [(pt.x, pt.y) for pt in gdf_proj.geometry]
        
        # src.sample renvoie un générateur, on le convertit en liste
        results_generator = src.sample(coords, indexes=band_number)
        
        # Chaque résultat est un array, on extrait la valeur
        values = [item[0] for item in results_generator]
        return values

In [11]:
def get_descriptive_stats(df, column_name):
    """
    Calcule et affiche les statistiques descriptives pour une colonne numérique,
    groupées par la colonne 'type_ilot'.
    
    Args:
        df (pd.DataFrame): Le DataFrame contenant les données.
        column_name (str): Le nom de la colonne à analyser.
    
    Returns:
        pd.DataFrame: Un DataFrame contenant les statistiques descriptives.
    """
    # Vérifier si la colonne existe
    if column_name not in df.columns:
        print(f"ERREUR : La colonne '{column_name}' n'existe pas.")
        return None
    
    # 1. Copier les données pour ne pas modifier le DataFrame original
    stats_df = df[['type_ilot', column_name]].copy()
    
    # 2. S'assurer que la colonne est de type numérique, convertir les erreurs en NaN
    stats_df[column_name] = pd.to_numeric(stats_df[column_name], errors='coerce')
    
    # 3. Supprimer les lignes avec des valeurs NaN pour éviter les erreurs de calcul
    initial_rows = len(stats_df)
    stats_df.dropna(subset=[column_name], inplace=True)
    if len(stats_df) < initial_rows:
        print(f"Note : {initial_rows - len(stats_df)} ligne(s) avec des valeurs non valides ont été ignorées.")
        
    # 4. Calculer les statistiques descriptives par 'type_ilot'
    descriptive_stats = stats_df.groupby('type_ilot')[column_name].describe()
    
    return descriptive_stats

---

## 4. Extraction depuis OpenLandMap

Les données de cette section sont extraites de la base de données **OpenLandMap**, qui fournit des prédictions globales sur les propriétés du sol. Les couches de données sont disponibles pour plusieurs profondeurs standards, dont **0-30 cm**, **30-60 cm** et **60-100 cm**. Il s'agit de cartes mondiales des propriétés du sol produites par le [Land & Carbon Lab](https://landcarbonlab.org/) pour la période 2000-2022.Nous utiliserons les profondeurs **0-30 cm** et **30-60 cm** pour la periode **2020-2022**.

### 4.1. Teneur en argile (%, Clay Content)

L'argile est la fraction la plus fine des particules du sol, avec un **diamètre inférieur à 0,002 mm**. Ce sont de petites particules à charge négative capables de capter et de retenir les nutriments (à charge positive) ainsi que de stocker l'eau. L'argile joue également un rôle de liant entre les particules de sable et de limon, contribuant à la structure du sol.

* **0-30 cm :** *[Teneur Argile 0-30](https://zenodo.org/records/15528401)*
* **30-60 cm :** *[Teneur Argile 30-60](https://zenodo.org/records/15528405)*

In [22]:
# 1. Definir les chemins URL vers les rasters
CLAY_COG_URL_0_30 = "https://zenodo.org/records/15528401/files/clay.tot_iso.11277.2020.wpct_m_120m_b0cm..30cm_20200101_20221231_g_epsg.4326_v20250523.tif"
CLAY_COG_URL_30_60 = "https://zenodo.org/records/15528405/files/clay.tot_iso.11277.2020.wpct_m_120m_b30cm..60cm_20200101_20221231_g_epsg.4326_v20250523.tif"

# --- Extraction de la teneur en argile ---
print("--- Extraction de la teneur en argile ---")
clay_values_0_30 = extract_values(CLAY_COG_URL_0_30, gdf_points)
clay_values_30_60 = extract_values(CLAY_COG_URL_30_60, gdf_points)

# Ajouter les résultats au GeoDataFrame avec les bons noms de colonnes
gdf_points['ARG1'] = clay_values_0_30
gdf_points['ARG2'] = clay_values_30_60
print("\nTraitement terminé.")

# Afficher un aperçu des nouvelles colonnes
print("\nAperçu des résultats pour l'argile :")
display(gdf_points[['ZONE_PEDO', 'ARG1', 'ARG2']].head())

# --- MODIFICATION : Afficher les dimensions au lieu de sauvegarder ---
print(f"\nDimensions actuelles du DataFrame : {gdf_points.shape[0]} lignes et {gdf_points.shape[1]} colonnes.")

--- Extraction de la teneur en argile ---
Extraction des valeurs depuis : https://zenodo.org/records/15528401/files/clay.tot_iso.11277.2020.wpct_m_120m_b0cm..30cm_20200101_20221231_g_epsg.4326_v20250523.tif (Bande: 1)
Extraction des valeurs depuis : https://zenodo.org/records/15528405/files/clay.tot_iso.11277.2020.wpct_m_120m_b30cm..60cm_20200101_20221231_g_epsg.4326_v20250523.tif (Bande: 1)

Traitement terminé.

Aperçu des résultats pour l'argile :


,ZONE_PEDO,ARG1,ARG2
0,dior_cc_avec_arbr,18,18
1,dekk_cb_avec_arbr,18,19
2,dior_cb_avec_arbr,18,19
3,dior_cb_avec_arbr,18,19
4,dior_cb_avec_arbr,18,19



Dimensions actuelles du DataFrame : 749 lignes et 14 colonnes.


### 4.2. Teneur en sable (%, Sand Content)

Le sable constitue la fraction la plus grossière des particules minérales du sol, avec un **diamètre compris entre 0,05 et 2 mm**. Contrairement à l'argile, ces particules sont plus grosses et chimiquement peu actives. Elles assurent une bonne aération et un bon drainage du sol mais retiennent faiblement l'eau et les nutriments, contribuant à une texture légère.

* **0-30 cm :** *[Teneur Sable 0-30](https://zenodo.org/records/15528413)*
* **30-60 cm :** *[Teneur Sable 30-60](https://zenodo.org/records/15528417)*

In [23]:
# Definir les chemins URL vers les rasters
SAND_COG_URL_0_30 = "https://zenodo.org/records/15528413/files/sand.tot_iso.11277.2020.wpct_m_120m_b0cm..30cm_20200101_20221231_g_epsg.4326_v20250523.tif"
SAND_COG_URL_30_60 = "https://zenodo.org/records/15528417/files/sand.tot_iso.11277.2020.wpct_m_120m_b30cm..60cm_20200101_20221231_g_epsg.4326_v20250523.tif"

# --- Extraction de la teneur en sable ---
print("--- Extraction de la teneur en sable ---")
sand_values_0_30 = extract_values(SAND_COG_URL_0_30, gdf_points)
sand_values_30_60 = extract_values(SAND_COG_URL_30_60, gdf_points)

# Ajouter les résultats au GeoDataFrame avec les bons noms de colonnes
gdf_points['SAB1'] = sand_values_0_30
gdf_points['SAB2'] = sand_values_30_60
print("\nTraitement terminé.")

# Afficher un aperçu des nouvelles colonnes
print("\nAperçu des résultats pour le sable :")
display(gdf_points[['ZONE_PEDO', 'SAB1', 'SAB2']].head())

# --- MODIFICATION : Afficher les dimensions au lieu de sauvegarder ---
print(f"\nDimensions actuelles du DataFrame : {gdf_points.shape[0]} lignes et {gdf_points.shape[1]} colonnes.")

--- Extraction de la teneur en sable ---
Extraction des valeurs depuis : https://zenodo.org/records/15528413/files/sand.tot_iso.11277.2020.wpct_m_120m_b0cm..30cm_20200101_20221231_g_epsg.4326_v20250523.tif (Bande: 1)
Extraction des valeurs depuis : https://zenodo.org/records/15528417/files/sand.tot_iso.11277.2020.wpct_m_120m_b30cm..60cm_20200101_20221231_g_epsg.4326_v20250523.tif (Bande: 1)

Traitement terminé.

Aperçu des résultats pour le sable :


,ZONE_PEDO,SAB1,SAB2
0,dior_cc_avec_arbr,65,66
1,dekk_cb_avec_arbr,66,65
2,dior_cb_avec_arbr,65,65
3,dior_cb_avec_arbr,66,66
4,dior_cb_avec_arbr,66,66



Dimensions actuelles du DataFrame : 749 lignes et 16 colonnes.


### 4.3. Densité apparente (kg/m³, Bulk Density)

La densité apparente mesure la masse de sol sec par unité de volume total, incluant les espaces poreux. C'est un indicateur clé de la **compaction du sol**. Une densité élevée implique moins d'espace pour l'air et l'eau, ce qui peut limiter la pénétration des racines et l'infiltration de l'eau. Pour être utilisées dans le modèle MAELIA, ces données seront converties en g/cm³. 

**NB : Les liens ci-dessous sont des liens de telechargements directs.**

* **0-30 cm :** *[Densite Apparente 0-30](https://s3.opengeohub.org/global-soil/global_soil_props_v20250204_mosaics/bd.core_iso.11272.2017.g.cm3_m_30m_b0cm..30cm_20200101_20221231_g_epsg.4326_v20250204.tif)*
* **30-60 cm :** *[Densite Apparente 0-30](https://s3.opengeohub.org/global-soil/global_soil_props_v20250204_mosaics/bd.core_iso.11272.2017.g.cm3_m_30m_b30cm..60cm_20200101_20221231_g_epsg.4326_v20250204.tif)*

In [24]:
# 1. Definir les chemins URL vers les rasters
BD_COG_URL_0_30 = "https://s3.opengeohub.org/global-soil/global_soil_props_v20250204_mosaics/bd.core_iso.11272.2017.g.cm3_m_30m_b0cm..30cm_20200101_20221231_g_epsg.4326_v20250204.tif"
BD_COG_URL_30_60 = "https://s3.opengeohub.org/global-soil/global_soil_props_v20250204_mosaics/bd.core_iso.11272.2017.g.cm3_m_30m_b30cm..60cm_20200101_20221231_g_epsg.4326_v20250204.tif"

# 2. Appliquer la fonction d'extraction
print("--- Extraction de la densité apparente (en kg/m³) ---")
bd_values_kgm3_0_30 = extract_values(BD_COG_URL_0_30, gdf_points)
bd_values_kgm3_30_60 = extract_values(BD_COG_URL_30_60, gdf_points)

# 3. CONVERSION D'UNITÉS (kg/m³ -> g/cm³)
# On divise par 1000, en gérant les valeurs 'None' si un point est hors raster.
bd_values_gcm3_0_30 = [val / 1000 if val is not None else None for val in bd_values_kgm3_0_30]
bd_values_gcm3_30_60 = [val / 1000 if val is not None else None for val in bd_values_kgm3_30_60]
print("\nConversion des unités en g/cm³ effectuée.")

# 4. Ajouter les résultats convertis au GeoDataFrame
gdf_points['DAH1'] = bd_values_gcm3_0_30
gdf_points['DAH2'] = bd_values_gcm3_30_60
print("\nTraitement terminé.")

# Afficher un aperçu des nouvelles colonnes
print("\nAperçu des résultats pour la densité apparente :")
display(gdf_points[['ZONE_PEDO', 'DAH1', 'DAH2']].head())

# Afficher les dimensions actuelles du DataFrame
print(f"\nDimensions actuelles du DataFrame : {gdf_points.shape[0]} lignes et {gdf_points.shape[1]} colonnes.")

--- Extraction de la densité apparente (en kg/m³) ---
Extraction des valeurs depuis : https://s3.opengeohub.org/global-soil/global_soil_props_v20250204_mosaics/bd.core_iso.11272.2017.g.cm3_m_30m_b0cm..30cm_20200101_20221231_g_epsg.4326_v20250204.tif (Bande: 1)
Extraction des valeurs depuis : https://s3.opengeohub.org/global-soil/global_soil_props_v20250204_mosaics/bd.core_iso.11272.2017.g.cm3_m_30m_b30cm..60cm_20200101_20221231_g_epsg.4326_v20250204.tif (Bande: 1)

Conversion des unités en g/cm³ effectuée.

Traitement terminé.

Aperçu des résultats pour la densité apparente :


,ZONE_PEDO,DAH1,DAH2
0,dior_cc_avec_arbr,0.154,0.149
1,dekk_cb_avec_arbr,0.155,0.152
2,dior_cb_avec_arbr,0.154,0.152
3,dior_cb_avec_arbr,0.153,0.150
4,dior_cb_avec_arbr,0.155,0.151



Dimensions actuelles du DataFrame : 749 lignes et 18 colonnes.


### 4.4. Teneur en carbone (g/kg, Carbon Content)

La teneur en carbone organique est une composante majeure de la matière organique et un indicateur clé de la **santé et de la fertilité des sols**. Elle influence la structure du sol, sa capacité de rétention en eau et en nutriments, et sert de source d'énergie pour les micro-organismes du sol. Les donnees extraites sont en g/kg. Elles seront converties en pourcentage puis multipliees par le facteur de Van Bemmelen, **1,724**, afin d'obtenir la teneur en matiere organique.

**NB : Les liens ci-dessous sont des liens de telechargements directs.**

* **0-30 cm :** *[Teneur en Carbone (g/kg) 0 - 30](https://s3.opengeohub.org/global-soil/global_soil_props_v20250204_mosaics/oc_iso.10694.1995.wpml_m_30m_b0cm..30cm_20200101_20221231_g_epsg.4326_v20250204.tif)*
* **30-60 cm :** *[Teneur en Carbone (g/kg) 30 - 60](https://s3.opengeohub.org/global-soil/global_soil_props_v20250204_mosaics/oc_iso.10694.1995.wpml_m_30m_b30cm..60cm_20200101_20221231_g_epsg.4326_v20250204.tif)*

In [25]:
# 1. Definir les chemins URL vers les rasters
SOC_COG_URL_0_30 = "https://s3.opengeohub.org/global-soil/global_soil_props_v20250204_mosaics/oc_iso.10694.1995.wpml_m_30m_b0cm..30cm_20200101_20221231_g_epsg.4326_v20250204.tif"
SOC_COG_URL_30_60 = "https://s3.opengeohub.org/global-soil/global_soil_props_v20250204_mosaics/oc_iso.10694.1995.wpml_m_30m_b30cm..60cm_20200101_20221231_g_epsg.4326_v20250204.tif"

# 2. Appliquer la fonction d'extraction (valeurs en g/kg)
print("--- Extraction de la teneur en carbone (en g/kg) ---")
soc_values_gkg_0_30 = extract_values(SOC_COG_URL_0_30, gdf_points)
soc_values_gkg_30_60 = extract_values(SOC_COG_URL_30_60, gdf_points)

# 3. CONVERSIONS ET CALCULS
# 3a. Convertir le carbone de g/kg en % (diviser par 10)
soc_values_pct_0_30 = [val / 10 if val is not None else None for val in soc_values_gkg_0_30]
soc_values_pct_30_60 = [val / 10 if val is not None else None for val in soc_values_gkg_30_60]

# 3b. Calculer la Matière Organique en % (Carbone % * 1.724)
mo_values_pct_0_30 = [val * 1.724 if val is not None else None for val in soc_values_pct_0_30]
mo_values_pct_30_60 = [val * 1.724 if val is not None else None for val in soc_values_pct_30_60]
print("\nConversion des unités et calcul de la M.O. terminés.")

# 4. Ajouter les 4 nouvelles colonnes au GeoDataFrame
gdf_points['C1'] = soc_values_pct_0_30
gdf_points['C2'] = soc_values_pct_30_60
gdf_points['MO1'] = mo_values_pct_0_30
gdf_points['MO2'] = mo_values_pct_30_60
print("\nTraitement terminé.")

# Afficher un aperçu des nouvelles colonnes
print("\nAperçu des résultats pour le carbone et la matière organique :")
display(gdf_points[['ZONE_PEDO', 'C1', 'MO1', 'C2', 'MO2']].head())

# Afficher les dimensions actuelles du DataFrame
print(f"\nDimensions actuelles du DataFrame : {gdf_points.shape[0]} lignes et {gdf_points.shape[1]} colonnes.")

--- Extraction de la teneur en carbone (en g/kg) ---
Extraction des valeurs depuis : https://s3.opengeohub.org/global-soil/global_soil_props_v20250204_mosaics/oc_iso.10694.1995.wpml_m_30m_b0cm..30cm_20200101_20221231_g_epsg.4326_v20250204.tif (Bande: 1)
Extraction des valeurs depuis : https://s3.opengeohub.org/global-soil/global_soil_props_v20250204_mosaics/oc_iso.10694.1995.wpml_m_30m_b30cm..60cm_20200101_20221231_g_epsg.4326_v20250204.tif (Bande: 1)

Conversion des unités et calcul de la M.O. terminés.

Traitement terminé.

Aperçu des résultats pour le carbone et la matière organique :


,ZONE_PEDO,C1,MO1,C2,MO2
0,dior_cc_avec_arbr,5.4,9.3096,3.7,6.3788
1,dekk_cb_avec_arbr,4.1,7.0684,3.0,5.1720
2,dior_cb_avec_arbr,4.3,7.4132,3.0,5.1720
3,dior_cb_avec_arbr,4.6,7.9304,3.0,5.1720
4,dior_cb_avec_arbr,4.3,7.4132,3.0,5.1720



Dimensions actuelles du DataFrame : 749 lignes et 22 colonnes.


In [26]:
# Sauvegarder le DataFrame complet avec les données d'OpenLandMap
output_csv_path.parent.mkdir(parents=True, exist_ok=True)
gdf_points.to_csv(output_csv_path, index=False, sep=',')

print("✅ DataFrame avec les données d'OpenLandMap sauvegardé avec succès.")
print(f"   -> Emplacement : {output_csv_path}")

✅ DataFrame avec les données d'OpenLandMap sauvegardé avec succès.
   -> Emplacement : C:\Users\Cheikhou\Desktop\Ferlo_Sine\maelia-data-diohine-v1\data\sols\csv\processed\donnees_typesDeSol.csv


### Bilan des Extractions OpenLandMap

Dans cette section du notebook, nous avons chargé les points d'échantillonnage préparés précédemment. En utilisant des fonctions utilitaires, nous avons enrichi ces points avec les données pédologiques de la base de données **OpenLandMap**.

Les propriétés suivantes ont été extraites pour les deux horizons de sol (0-30 cm et 30-60 cm) :
* Teneur en **argile** (`ARG1`, `ARG2`)
* Teneur en **sable** (`SAB1`, `SAB2`)
* **Densité apparente** (`DAH1`, `DAH2`), avec une conversion d'unités (kg/m³ → g/cm³).
* **Carbone Organique** (`C1`, `C2`), qui a servi à calculer la **Matière Organique** (`MO1`, `MO2`) via le facteur de Van Bemmelen.

Le DataFrame `gdf_points` contient maintenant l'ensemble de ces nouvelles variables. Le résultat final a été sauvegardé pour être utilisé dans les prochaines étapes.

## 5. Extraction depuis iSDA Africa (API)

Pour les propriétés chimiques du sol comme le pH et l'azote, les données ont été extraites via l'API (Interface de Programmation) de **iSDA Africa**. Cette source fournit des données à une résolution spatiale de **30 mètres** pour les profondeurs **0-20 cm** et **20-50 cm**.

### 5.1. Processus d'Authentification

L'accès à l'API iSDA nécessite une authentification pour chaque session. Ce processus, géré directement dans le notebook, se déroule en deux temps :

1.  **Inscription :** Il est nécessaire de créer un compte sur le portail des développeurs d'iSDA ([https://www.isda-africa.com/isdasoil/developer/](https://www.isda-africa.com/isdasoil/developer/)) pour obtenir un nom d'utilisateur et un mot de passe.
2.  **Obtention du Token :** Le script envoie une requête d'authentification avec ces identifiants pour recevoir un `access token`. Ce token temporaire est ensuite inclus dans toutes les requêtes suivantes pour autoriser l'accès aux données.

La méthodologie technique complète est détaillée dans le tutoriel officiel d'iSDA : [iSDAsoil-tutorial.ipynb](https://github.com/iSDA-Africa/isdasoil-tutorial/blob/main/iSDAsoil-tutorial.ipynb).

### 5.2. Harmonisation des Horizons de Sol (Calcul Pro-Rata)

Les horizons de sol fournis par iSDA (0-20 cm et 20-50 cm) ne correspondant pas à nos horizons cibles (0-30 cm et 30-60 cm), une harmonisation par calcul pro-rata a été appliquée :

* **Pour la couche cible 0-30 cm :** Une moyenne pondérée est calculée, composée de la valeur des 20 premiers centimètres (source 0-20 cm) et des 10 centimètres suivants (source 20-50 cm), via la fonction `calculer_horizon_0_30()`.
* **Pour la couche cible 30-60 cm :** En l'absence de données au-delà de 50 cm, nous avons posé l'hypothèse que les propriétés de la couche 20-50 cm se prolongent. La valeur de la couche 20-50 cm est donc directement assignée à notre couche cible 30-60 cm, via la fonction `calculer_horizon_30_60()`.

### 5.3. Teneur en azote total (g/kg)

La teneur en azote total est une donnée essentielle pour évaluer la fertilité du sol et pour calculer le rapport C/N, un indicateur clé de la dynamique de la matière organique.

**Transformation des données :** Les valeurs de ce raster sont fournies avec un facteur d'échelle. Pour obtenir la teneur réelle en **g/kg**, il est nécessaire de **diviser par 100** les valeurs brutes extraites du fichier.

* **0-20 cm :** Bande 1 du fichier
* **20-50 cm :** Bande 2 du fichier

Lien vers le fichier : *[Azote Total](https://isdasoil.s3.amazonaws.com/soil_data/nitrogen_total/nitrogen_total.tif)*

In [13]:
def calculer_horizon_0_30(val_0_20, val_20_50):
    """
    Calcule la valeur pro-rata pour l'horizon 0-30 cm à partir
    des horizons 0-20 cm et 20-50 cm.
    """
    # Si l'une des deux valeurs est manquante, on ne peut pas calculer
    if pd.isna(val_0_20) or pd.isna(val_20_50):
        return np.nan
    
    # Formule de la moyenne pondérée
    valeur_cible = ((val_0_20 * 20) + (val_20_50 * 10)) / 30
    return valeur_cible

def calculer_horizon_30_60(val_20_50):
    """
    Calcule la valeur pro-rata pour l'horizon 30-60 cm à partir
    de l'horizon 20-50 cm (par extension).
    """
    # Si la valeur est manquante, on retourne NaN
    if pd.isna(val_20_50):
        return np.nan
    
    # On étend simplement la valeur de la couche 20-50
    return val_20_50

In [29]:
# 1. Definir le chemin URL vers le raster
N_COG_URL = "https://isdasoil.s3.amazonaws.com/soil_data/nitrogen_total/nitrogen_total.tif"

# 2. Appliquer la fonction d'extraction pour les deux bandes
print("--- Extraction de la teneur en azote (brute) ---")
# On corrige le nom du GeoDataFrame en gdf_points
n_values_raw_0_20 = extract_values(N_COG_URL, gdf_points, band_number=1)
n_values_raw_20_50 = extract_values(N_COG_URL, gdf_points, band_number=2)

# 3. Appliquer le facteur d'échelle (diviser par 100 pour obtenir des g/kg)
print("\nApplication du facteur d'échelle...")
n_values_gkg_0_20 = [val / 100 if val is not None else None for val in n_values_raw_0_20]
n_values_gkg_20_50 = [val / 100 if val is not None else None for val in n_values_raw_20_50]

# 4. Appliquer l'harmonisation pro-rata pour obtenir les horizons 0-30 et 30-60
print("Harmonisation des horizons de sol (pro-rata)...")
n_values_0_30 = [calculer_horizon_0_30(v1, v2) for v1, v2 in zip(n_values_gkg_0_20, n_values_gkg_20_50)]
n_values_30_60 = [calculer_horizon_30_60(v) for v in n_values_gkg_20_50]

# 5. Ajouter les résultats finaux au GeoDataFrame
# Nous les nommerons N1 et N2 en prévision du calcul C/N
gdf_points['N1'] = n_values_0_30
gdf_points['N2'] = n_values_30_60
print("\nTraitement terminé.")

# Afficher un aperçu des nouvelles colonnes
print("\nAperçu des résultats pour l'azote (en g/kg) :")
display(gdf_points[['ZONE_PEDO', 'N1', 'N2']].head())

# Afficher les dimensions actuelles du DataFrame
print(f"\nDimensions actuelles du DataFrame : {gdf_points.shape[0]} lignes et {gdf_points.shape[1]} colonnes.")

--- Extraction de la teneur en azote (brute) ---
Extraction des valeurs depuis : https://isdasoil.s3.amazonaws.com/soil_data/nitrogen_total/nitrogen_total.tif (Bande: 1)
Extraction des valeurs depuis : https://isdasoil.s3.amazonaws.com/soil_data/nitrogen_total/nitrogen_total.tif (Bande: 2)

Application du facteur d'échelle...
Harmonisation des horizons de sol (pro-rata)...

Traitement terminé.

Aperçu des résultats pour l'azote (en g/kg) :


,ZONE_PEDO,N1,N2
0,dior_cc_avec_arbr,0.343333,0.31
1,dekk_cb_avec_arbr,0.356667,0.33
2,dior_cb_avec_arbr,0.340000,0.32
3,dior_cb_avec_arbr,0.296667,0.29
4,dior_cb_avec_arbr,0.336667,0.31



Dimensions actuelles du DataFrame : 749 lignes et 24 colonnes.


### 5.4. pH du sol

Le pH est une mesure de l'acidité ou de l'alcalinité du sol qui influence directement la disponibilité des nutriments pour les plantes.

Contrairement à l'azote où le fichier raster complet a été utilisé, nous employons ici l'**API d'iSDA** pour le pH. Cette approche illustre la seconde méthode d'accès aux données, qui consiste à requêter des valeurs précises directement pour nos points d'échantillonnage. Il faut noter qu'elle est plus lente que la premiere en raison des nombreux API calls.

Ce processus est géré par deux blocs de code :

1.  **Obtention du token** : Un premier script envoie les identifiants de l'utilisateur au serveur iSDA pour recevoir en retour un `access token`. Cette clé temporaire autorise les requêtes de données.

2.  **Extraction par une fonction** : La fonction `get_isda_property_reproject` utilise ce token pour interroger l'API. Elle prend en charge l'ensemble du processus :
    * Elle s'assure que le système de coordonnées des points est correct en le reprojetant en interne.
    * Elle boucle sur chaque point, envoie ses coordonnées à l'API et demande la valeur du pH pour les profondeurs **0-20 cm** et **20-50 cm**.
    * Elle retourne la liste des valeurs de pH, prête à être ajoutée au jeu de données.

In [33]:
import requests
from getpass import getpass

# Demander les identifiants de manière interactive
username = input("Entrez votre email iSDA : ")
password = getpass("Entrez votre mot de passe iSDA : ")

# --- Connexion à l'API ---
base_url = "https://api.isda-africa.com"
login_payload = {"username": username, "password": password}
access_token = None

# Vérifier que les identifiants ont bien été chargés
if not username or not password:
    print("🚨 ERREUR : L'identifiant ou le mot de passe n'a pas été trouvé.")
    print("   Veuillez vérifier que le fichier .env existe à la racine du projet et contient les variables ISDA_USERNAME et ISDA_PASSWORD.")
else:
    try:
        print(f"Connexion à {base_url}/login pour obtenir le token...")
        response = requests.post(f"{base_url}/login", data=login_payload)
        response.raise_for_status() # Lève une erreur en cas d'échec
        access_token = response.json().get("access_token")
        print("✅ Token obtenu avec succès !")
    except Exception as e:
        print(f"ERREUR lors de l'obtention du token : {e}")

Entrez votre email iSDA :  cheikhou.kane@urdfs.edu.sn
Entrez votre mot de passe iSDA :  ········


Connexion à https://api.isda-africa.com/login pour obtenir le token...
✅ Token obtenu avec succès !


In [34]:
def get_isda_property_reproject(gdf_source, token, property_name, depth="0-20"):
    """
    Reprojette une copie du GeoDataFrame en interne, puis interroge l'API iSDA
    pour obtenir la valeur d'une propriété de sol spécifique.
    """
    if not token:
        print("Échec : Token manquant.")
        return None

    # 1. Reprojeter une copie du GeoDataFrame en WGS84 (EPSG:4326)
    print("Reprojection interne en EPSG:4326...")
    gdf_wgs84 = gdf_source.to_crs(epsg=4326)

    # 2. Préparer l'extraction
    base_url = "https://api.isda-africa.com"
    query_url = f"{base_url}/isdasoil/v2/soilproperty"
    headers = {"Authorization": f"Bearer {token}"}
    
    valeurs_extraites = []

    print(f"Début de la récupération pour : '{property_name}'...")
    # 3. Boucle sur le GeoDataFrame reprojeté
    for index, row in gdf_wgs84.iterrows():
        params = {
            "lat": row.geometry.y,
            "lon": row.geometry.x,
            "property": property_name,
            "depth": depth
        }
        
        try:
            response = requests.get(query_url, headers=headers, params=params)
            response.raise_for_status()
            data = response.json()
            value = data['property'][property_name][0]['value']['value']
            valeurs_extraites.append(value)
        except Exception:
            valeurs_extraites.append(None)
        
        if (index + 1) % 50 == 0:
            print(f"  ... {index + 1} / {len(gdf_wgs84)} points traités.")
        time.sleep(0.05)

    return valeurs_extraites

print("Fonction 'get_isda_property_reproject' définie.")

Fonction 'get_isda_property_reproject' définie.


In [35]:
# Vérifier que le token d'accès est bien disponible
if access_token:
    # --- Paramètres ---
    propriete_a_recuperer = 'ph'
    
    # --- Extraction pour la profondeur 0-20 cm ---
    print(f"--- Début de l'extraction pour la propriété '{propriete_a_recuperer}' (0-20 cm) ---")
    ph_values_0_20 = get_isda_property_reproject(
        gdf_points, 
        access_token, 
        propriete_a_recuperer,
        depth="0-20"
    )

    # --- Extraction pour la profondeur 20-50 cm ---
    print(f"\n--- Début de l'extraction pour la propriété '{propriete_a_recuperer}' (20-50 cm) ---")
    ph_values_20_50 = get_isda_property_reproject(
        gdf_points, 
        access_token, 
        propriete_a_recuperer,
        depth="20-50"
    )

    # --- Harmonisation Pro-Rata (sans division préalable) ---
    print("\n--- Harmonisation des horizons de sol (pro-rata) ---")
    ph_values_0_30 = [calculer_horizon_0_30(v1, v2) for v1, v2 in zip(ph_values_0_20, ph_values_20_50)]
    ph_values_30_60 = [calculer_horizon_30_60(v) for v in ph_values_20_50]

    # --- Ajout des résultats au GeoDataFrame ---
    gdf_points['PH1'] = ph_values_0_30
    gdf_points['PH2'] = ph_values_30_60
    
    print(f"\n✅ Récupération et harmonisation de '{propriete_a_recuperer}' terminées !")
    print("\nAperçu des nouvelles colonnes 'PH1' et 'PH2':")
    display(gdf_points[['ZONE_PEDO', 'PH1', 'PH2']].head())

else:
    print("🚨 Échec : Aucun access token disponible. Veuillez exécuter la cellule d'authentification.")

--- Début de l'extraction pour la propriété 'ph' (0-20 cm) ---
Reprojection interne en EPSG:4326...
Début de la récupération pour : 'ph'...
  ... 50 / 749 points traités.
  ... 100 / 749 points traités.
  ... 150 / 749 points traités.
  ... 200 / 749 points traités.
  ... 250 / 749 points traités.
  ... 300 / 749 points traités.
  ... 350 / 749 points traités.
  ... 400 / 749 points traités.
  ... 450 / 749 points traités.
  ... 500 / 749 points traités.
  ... 550 / 749 points traités.
  ... 600 / 749 points traités.
  ... 650 / 749 points traités.
  ... 700 / 749 points traités.

--- Début de l'extraction pour la propriété 'ph' (20-50 cm) ---
Reprojection interne en EPSG:4326...
Début de la récupération pour : 'ph'...
  ... 50 / 749 points traités.
  ... 100 / 749 points traités.
  ... 150 / 749 points traités.
  ... 200 / 749 points traités.
  ... 250 / 749 points traités.
  ... 300 / 749 points traités.
  ... 350 / 749 points traités.
  ... 400 / 749 points traités.
  ... 450 / 749 

,ZONE_PEDO,PH1,PH2
0,dior_cc_avec_arbr,5.866667,5.8
1,dekk_cb_avec_arbr,5.766667,5.7
2,dior_cb_avec_arbr,5.966667,5.9
3,dior_cb_avec_arbr,5.966667,5.9
4,dior_cb_avec_arbr,5.866667,5.8


## 5.5. Bilan des Extractions iSDA Africa

Dans cette section, nous avons enrichi nos points d'échantillonnage avec les données de la source **iSDA Africa**.

* Les propriétés de **teneur en azote** (`N`) et de **pH** ont été récupérées pour les deux horizons de sol.
* Deux méthodes d'accès ont été mises en œuvre : le téléchargement d'un raster multi-bandes pour l'azote et des appels à l'**API** pour le pH, qui ont nécessité une authentification sécurisée via un token.
* Un défi majeur a été l'**harmonisation des horizons de sol**. Les données sources (0-20 cm, 20-50 cm) ont été converties à nos horizons cibles (0-30 cm, 30-60 cm) en utilisant des fonctions de calcul **pro-rata** (moyenne pondérée).
* Après application des transformations nécessaires (notamment un facteur d'échelle pour l'azote), les nouvelles variables `N1`, `N2`, `PH1` et `PH2` ont été ajoutées au DataFrame principal.

In [37]:
# Sauvegarder le DataFrame complet
output_csv_path.parent.mkdir(parents=True, exist_ok=True)
gdf_points.to_csv(output_csv_path, index=False, sep=',')

print("✅ DataFrame avec les données d'iSDA Africa sauvegardé avec succès.")
print(f"   -> Emplacement : {output_csv_path}")

✅ DataFrame avec les données d'iSDA Africa sauvegardé avec succès.
   -> Emplacement : C:\Users\Cheikhou\Desktop\Ferlo_Sine\maelia-data-diohine-v1\data\sols\csv\processed\donnees_typesDeSol.csv


## 6. Extraction depuis Cirad Dataverse (Propriétés Hydriques)

Les données sur les propriétés hydriques du sol proviennent d'un jeu de données à haute résolution (30m) hébergé sur le portail **Cirad Dataverse**. Ces cartes ont été créées en appliquant le modèle **USDA Rosetta3** aux couches de texture du sol et de densité apparente d'iSDA Africa pour en déduire les propriétés de rétention en eau. Les profondeurs disponibles sont **0-20 cm** et **20-50 cm**.

Pour cette analyse, six fichiers raster compressés (`.tif.gz`) ont été téléchargés localement. Ils nous permettront d'extraire les variables suivantes :

* L'humidité volumique à la **capacité au champ**.
* L'humidité volumique au **point de flétrissement**.
* La **réserve en eau utile**, qui sera calculée à partir des deux variables précédentes et de l'épaisseur de la couche de sol.

Lien vers le jeu de données : [DOI: 10.18167/DVN1/SGNSII](https://dataverse.cirad.fr/dataset.xhtml?persistentId=doi:10.18167/DVN1/SGNSII)

Pour lire ces fichiers locaux compressés, la fonction `extract_values_local_gz` a été utilisée. Elle ouvre un fichier `.tif.gz`, s'assure que les points d'échantillonnage sont dans le bon système de coordonnées, puis extrait la valeur du raster correspondante pour chaque point.

In [1]:
def extract_values_local_gz(local_file_path, gdf, band_number=1):
    """
    Ouvre un fichier .tif.gz local et extrait les valeurs d'une bande
    spécifique pour chaque point d'un GeoDataFrame.
    """
    print(f"Lecture du fichier : {local_file_path.name} (Bande: {band_number})")
    
    with gzip.open(local_file_path, 'rb') as gz_f:
        with rasterio.open(gz_f) as src:
            gdf_proj = gdf.to_crs(src.crs)
            values = []
            for pt in gdf_proj.geometry:
                try:
                    row, col = src.index(pt.x, pt.y)
                    window = Window(col, row, 1, 1)
                    
                    # Utilise le paramètre band_number pour lire la bonne bande
                    data = src.read(band_number, window=window)
                    
                    values.append(data[0, 0])
                except IndexError:
                    values.append(None) # Gère le cas où le point est hors du raster
    
    return values

### 6.1. Humidité volumique à la capacité au champ

C'est la quantité d'eau que le sol peut retenir après que l'excès d'eau a été drainé par la gravité. Elle représente la limite supérieure de l'eau disponible pour les plantes.

* **0-20 cm :** `senegal_theta_fc_0_20cm_mask.tif.gz`
* **20-50 cm :** `senegal_theta_fc_20_50cm_mask.tif.gz`

Lien vers le jeu de données global : [DOI: 10.18167/DVN1/SGNSII](https://dataverse.cirad.fr/dataset.xhtml?persistentId=doi:10.18167/DVN1/SGNSII)

In [11]:
base_dir = Path.cwd().parent.resolve()
input_csv_path = base_dir / "data" / "sols" / "csv" / "processed" / "donnees_typesDeSol.csv"
SOURCE_CRS = "epsg:32628"
try:
    gdf_points = load_geodata_from_wkt(
        file_path=input_csv_path,
        geometry_col='geometry',
        source_crs=SOURCE_CRS
    )
    print("✅ GeoDataFrame complet chargé avec succès.")
    print(f"   -> Contient {gdf_points.shape[0]} points et {gdf_points.shape[1]} colonnes.")
    display(gdf_points.head())

except Exception as e:
    print(f"🚨 ERREUR : {e}")

Chargement des données depuis : C:\Users\Cheikhou\Desktop\Ferlo_Sine\maelia-data-diohine-v1\data\sols\csv\processed\donnees_typesDeSol.csv
✅ GeoDataFrame complet chargé avec succès.
   -> Contient 749 points et 26 colonnes.


,parcel_id,N°_PARCEL,TYP_SOL,Arbre,Long,Lat,X_Centroid,Y_Centroid,geometry,Type_champ,...,DAH1,DAH2,C1,C2,MO1,MO2,N1,N2,PH1,PH2
0,b7698961-a77b-4fb2-8c94-2f090c1d6fd2,147.0,Dior,1,337382.976620,1.603607e+06,337365.773977,1.603611e+06,POINT (337382.977 1603607.337),CC,...,0.154,0.149,5.4,3.7,9.3096,6.3788,0.343333,0.31,5.866667,5.8
1,dae3c8e5-9f33-4354-826d-aed699b6c17c,149.0,Dekk,1,336145.755091,1.603489e+06,336099.653300,1.603497e+06,POINT (336145.755 1603488.988),CB,...,0.155,0.152,4.1,3.0,7.0684,5.1720,0.356667,0.33,5.766667,5.7
2,e85b0517-6fda-4422-95a3-571360932f23,150.0,Dior,1,336296.074335,1.603034e+06,336302.866304,1.603039e+06,POINT (336296.074 1603033.949),CB,...,0.154,0.152,4.3,3.0,7.4132,5.1720,0.340000,0.32,5.966667,5.9
3,d28eb822-617a-4498-acbf-4f919ee1e3ae,144.0,Dior,1,335917.410954,1.602738e+06,335886.039020,1.602729e+06,POINT (335917.411 1602738.353),CB,...,0.153,0.150,4.6,3.0,7.9304,5.1720,0.296667,0.29,5.966667,5.9
4,2a2005cb-29a4-48bd-b7ad-fca2bc44db44,148.0,Dior,1,336894.131809,1.603453e+06,336894.334621,1.603456e+06,POINT (336894.132 1603453.182),CB,...,0.155,0.151,4.3,3.0,7.4132,5.1720,0.336667,0.31,5.866667,5.8


In [17]:
# 1. CONFIGURATION
#    Définir tous les fichiers à traiter dans une liste de dictionnaires.
fichiers_a_traiter = [
    {'nom_fichier': 'senegal_theta_fc_0_20cm_mask.tif.gz',  'col_temp': 'hcc_0_20'},
    {'nom_fichier': 'senegal_theta_fc_20_50cm_mask.tif.gz',  'col_temp': 'hcc_20_50'},
    {'nom_fichier': 'senegal_theta_wp_0_20cm_mask.tif.gz', 'col_temp': 'hpfp_0_20'},
    {'nom_fichier': 'senegal_theta_wp_20_50cm_mask.tif.gz', 'col_temp': 'hpfp_20_50'},
]
dossier_raster_raw = base_dir / "data" / "sols" / "raster" / "raw"

# 2. BOUCLE D'EXTRACTION
print("--- Début de l'extraction des données hydriques (Cirad Dataverse) ---")
for item in fichiers_a_traiter:
    chemin_fichier = dossier_raster_raw / item['nom_fichier']
    nom_colonne = item['col_temp']
    try:
        valeurs = extract_values_local_gz(chemin_fichier, gdf_points)
        gdf_points[nom_colonne] = valeurs
        print(f"-> Succès ! La colonne '{nom_colonne}' a été ajoutée.")
    except Exception as e:
        print(f"-> 🚨 ERREUR pour le fichier {chemin_fichier.name} : {e}")

# 3. HARMONISATION (Pro-Rata sur les fractions)
print("\n--- Harmonisation des horizons (pro-rata) ---")
hcc_final_fraction1 = [calculer_horizon_0_30(v1, v2) for v1, v2 in zip(gdf_points['hcc_0_20'], gdf_points['hcc_20_50'])]
hcc_final_fraction2 = [calculer_horizon_30_60(v) for v in gdf_points['hcc_20_50']]
hpfp_final_fraction1 = [calculer_horizon_0_30(v1, v2) for v1, v2 in zip(gdf_points['hpfp_0_20'], gdf_points['hpfp_20_50'])]
hpfp_final_fraction2 = [calculer_horizon_30_60(v) for v in gdf_points['hpfp_20_50']]

# 4. CALCUL DE LA RÉSERVE UTILE (RUPRH)
print("\n--- Calcul de la Réserve Utile (RUPRH) ---")
epaisseur_mm = 300 # 30 cm
ruprh1 = [(hcc - hpfp) * epaisseur_mm if hcc is not None and hpfp is not None else None for hcc, hpfp in zip(hcc_final_fraction1, hpfp_final_fraction1)]
ruprh2 = [(hcc - hpfp) * epaisseur_mm if hcc is not None and hpfp is not None else None for hcc, hpfp in zip(hcc_final_fraction2, hpfp_final_fraction2)]

# 5. CRÉATION DES COLONNES FINALES (HCC/HPFP en %)
print("\n--- Création des colonnes finales au format MAELIA ---")
gdf_points['HCC1'] = [v * 100 if v is not None else None for v in hcc_final_fraction1]
gdf_points['HCC2'] = [v * 100 if v is not None else None for v in hcc_final_fraction2]
gdf_points['HPFP1'] = [v * 100 if v is not None else None for v in hpfp_final_fraction1]
gdf_points['HPFP2'] = [v * 100 if v is not None else None for v in hpfp_final_fraction2]
gdf_points['RUPRH1'] = ruprh1
gdf_points['RUPRH2'] = ruprh2

# 6. NETTOYAGE
print("\n--- Suppression des colonnes de travail intermédiaires ---")
colonnes_a_supprimer = [item['col_temp'] for item in fichiers_a_traiter]
gdf_points.drop(columns=colonnes_a_supprimer, inplace=True)

# 7. VÉRIFICATION FINALE
print("\n✅ Traitement des données hydriques terminé.")
print(f"Dimensions finales du DataFrame : {gdf_points.shape[0]} lignes et {gdf_points.shape[1]} colonnes.")
display(gdf_points[['ZONE_PEDO', 'HCC1', 'HPFP1', 'RUPRH1', 'HCC2', 'HPFP2', 'RUPRH2']].head())

--- Début de l'extraction des données hydriques (Cirad Dataverse) ---
Lecture du fichier : senegal_theta_fc_0_20cm_mask.tif.gz (Bande: 1)
-> Succès ! La colonne 'hcc_0_20' a été ajoutée.
Lecture du fichier : senegal_theta_fc_20_50cm_mask.tif.gz (Bande: 1)
-> Succès ! La colonne 'hcc_20_50' a été ajoutée.
Lecture du fichier : senegal_theta_wp_0_20cm_mask.tif.gz (Bande: 1)
-> Succès ! La colonne 'hpfp_0_20' a été ajoutée.
Lecture du fichier : senegal_theta_wp_20_50cm_mask.tif.gz (Bande: 1)
-> Succès ! La colonne 'hpfp_20_50' a été ajoutée.

--- Harmonisation des horizons (pro-rata) ---

--- Calcul de la Réserve Utile (RUPRH) ---

--- Création des colonnes finales au format MAELIA ---

--- Suppression des colonnes de travail intermédiaires ---

✅ Traitement des données hydriques terminé.
Dimensions finales du DataFrame : 749 lignes et 32 colonnes.


,ZONE_PEDO,HCC1,HPFP1,RUPRH1,HCC2,HPFP2,RUPRH2
0,dior_cc_avec_arbr,NaN,NaN,NaN,NaN,NaN,NaN
1,dekk_cb_avec_arbr,NaN,NaN,NaN,NaN,NaN,NaN
2,dior_cb_avec_arbr,NaN,NaN,NaN,18.988800,8.767632,30.663505
3,dior_cb_avec_arbr,NaN,NaN,NaN,NaN,NaN,NaN
4,dior_cb_avec_arbr,17.359354,8.044078,27.94583,17.490194,8.165084,27.975330


## Bilan des Extractions Cirad Dataverse

Cette section finale a permis d'intégrer les **propriétés hydriques** du sol, une étape clé de la caractérisation, en utilisant les données de **Cirad Dataverse**.

* Les données de base pour la **capacité au champ** (`HCC`) et le **point de flétrissement** (`HPFP`) ont été extraites des fichiers locaux compressés (`.tif.gz`).

* Un défi majeur a été l'harmonisation des horizons de sol. Les données sources (0-20 cm, 20-50 cm) ont été converties vers nos horizons cibles (0-30 cm, 30-60 cm) via un calcul **pro-rata** (moyenne pondérée) appliqué sur les valeurs d'humidité (fournies en tant que fractions, ex: 0.25).

* Une décision méthodologique importante a été prise pour la **Réserve Utile (`RUPRH`)**. Bien que des rasters pour cette variable soient disponibles dans la source de données, nous avons choisi de la **calculer** nous-mêmes. Cette approche est plus juste car elle applique la formule `(HCC - HPFP) * épaisseur` sur les valeurs de `HCC` et `HPFP` *après* leur harmonisation, garantissant ainsi la cohérence du calcul.

* Au final, les six variables hydriques (`HCC1`, `HCC2`, `HPFP1`, `HPFP2`, `RUPRH1`, `RUPRH2`) ont été créées et ajoutées au DataFrame, avec `HCC` et `HPFP` stockées en **pourcentage** pour être conformes au format MAELIA.

In [20]:
# --- Bilan final du DataFrame avant sauvegarde ---
print("--- Bilan final du DataFrame avant sauvegarde ---")
print(f"Dimensions : {gdf_points.shape[0]} lignes et {gdf_points.shape[1]} colonnes.")
print("\nListe des colonnes :")
print(gdf_points.columns.to_list())
# ----------------------------------------------------

# S'assurer que le dossier de sortie existe
output_csv_path.parent.mkdir(parents=True, exist_ok=True)

# Sauvegarder le DataFrame complet au format CSV
gdf_points.to_csv(output_csv_path, index=False, sep=',')

print(f"\n✅ DataFrame sauvegardé avec succès dans : {output_csv_path}")

--- Bilan final du DataFrame avant sauvegarde ---
Dimensions : 749 lignes et 32 colonnes.

Liste des colonnes :
['parcel_id', 'N°_PARCEL', 'TYP_SOL', 'Arbre', 'Long', 'Lat', 'X_Centroid', 'Y_Centroid', 'geometry', 'Type_champ', 'type_ilot', 'ZONE_PEDO', 'ARG1', 'ARG2', 'SAB1', 'SAB2', 'DAH1', 'DAH2', 'C1', 'C2', 'MO1', 'MO2', 'N1', 'N2', 'PH1', 'PH2', 'HCC1', 'HCC2', 'HPFP1', 'HPFP2', 'RUPRH1', 'RUPRH2']

✅ DataFrame sauvegardé avec succès dans : C:\Users\Cheikhou\Desktop\Ferlo_Sine\maelia-data-diohine-v1\data\sols\csv\processed\donnees_typesDeSol.csv


In [21]:
# 1. Définir la liste des 8 ZONE_PEDO valides
zones_pedo_valides = [
    'dior_cb_avec_arbr',
    'dior_cb_sans_arbr',
    'dior_cc_avec_arbr',
    'dior_cc_sans_arbr',
    'dekk_cb_avec_arbr',
    'dekk_cb_sans_arbr',
    'dekk/mbel_cb_avec_arbr',
    'dekk/mbel_cb_sans_arbr'
]

# 2. Afficher la situation avant le filtrage
print("--- Avant le nettoyage ---")
print(f"Nombre total de lignes : {len(gdf_points)}")
print("Répartition initiale des ZONE_PEDO :")
print(gdf_points['ZONE_PEDO'].value_counts())

--- Avant le nettoyage ---
Nombre total de lignes : 749
Répartition initiale des ZONE_PEDO :
ZONE_PEDO
dior_cb_sans_arbr         247
dior_cb_avec_arbr         193
dekk_cb_sans_arbr          83
dekk_cb_avec_arbr          63
dior_cc_sans_arbr          63
dior_cc_avec_arbr          55
dekk/mbel_cb_sans_arbr     20
dekk/mbel_cb_avec_arbr     15
dekk_cc_sans_arbr           3
dekk_cc_avec_arbr           2
dior/mbel_cb_avec_arbr      1
dekk/mbel_cc_avec_arbr      1
mbel_cb_sans_arbr           1
dior/mbel_cb_sans_arbr      1
dekk/mbel_cc_sans_arbr      1
Name: count, dtype: int64


In [22]:
# 3. Filtrer le DataFrame pour ne garder que les lignes valides
#    La méthode .isin() est parfaite pour cela.
gdf_points_nettoye = gdf_points[gdf_points['ZONE_PEDO'].isin(zones_pedo_valides)].copy()

# 4. Afficher la situation après le filtrage
print("\n--- Après le nettoyage ---")
print(f"Nombre de lignes conservées : {len(gdf_points_nettoye)}")
print(f"Nombre de lignes supprimées : {len(gdf_points) - len(gdf_points_nettoye)}")
print("\nRépartition finale des ZONE_PEDO :")
print(gdf_points_nettoye['ZONE_PEDO'].value_counts())


--- Après le nettoyage ---
Nombre de lignes conservées : 739
Nombre de lignes supprimées : 10

Répartition finale des ZONE_PEDO :
ZONE_PEDO
dior_cb_sans_arbr         247
dior_cb_avec_arbr         193
dekk_cb_sans_arbr          83
dekk_cb_avec_arbr          63
dior_cc_sans_arbr          63
dior_cc_avec_arbr          55
dekk/mbel_cb_sans_arbr     20
dekk/mbel_cb_avec_arbr     15
Name: count, dtype: int64


In [23]:
# Mettre à jour la variable principale pour les prochaines étapes
gdf_points = gdf_points_nettoye

# Sauvegarder la version nettoyée du fichier
output_csv_path.parent.mkdir(parents=True, exist_ok=True)
gdf_points.to_csv(output_csv_path, index=False, sep=',')

print("✅ DataFrame nettoyé sauvegardé avec succès.")
print(f"   -> Emplacement : {output_csv_path}")

✅ DataFrame nettoyé sauvegardé avec succès.
   -> Emplacement : C:\Users\Cheikhou\Desktop\Ferlo_Sine\maelia-data-diohine-v1\data\sols\csv\processed\donnees_typesDeSol.csv


In [24]:
# S'assurer que les colonnes à agréger sont bien de type numérique
# (liste non exhaustive, à compléter avec toutes vos colonnes de données)
colonnes_numeriques = ['ARG1', 'ARG2', 'SAB1', 'SAB2', 'DAH1', 'DAH2', 
                       'C1', 'C2', 'MO1', 'MO2', 'N1', 'N2', 
                       'PH1', 'PH2', 'HCC1', 'HCC2', 'HPFP1', 'HPFP2', 'RUPRH1', 'RUPRH2']

for col in colonnes_numeriques:
    gdf_points[col] = pd.to_numeric(gdf_points[col], errors='coerce')

# Grouper par ZONE_PEDO et calculer la moyenne pour chaque propriété
df_agrege = gdf_points.groupby('ZONE_PEDO')[colonnes_numeriques].mean().reset_index()

print("✅ Agrégation par moyenne terminée.")
print(f"   Le DataFrame de synthèse contient {df_agrege.shape[0]} lignes (une par ZONE_PEDO).")

print("\nAperçu de la table de synthèse finale :")
display(df_agrege.head())

✅ Agrégation par moyenne terminée.
   Le DataFrame de synthèse contient 8 lignes (une par ZONE_PEDO).

Aperçu de la table de synthèse finale :


,ZONE_PEDO,ARG1,ARG2,SAB1,SAB2,DAH1,DAH2,C1,C2,MO1,...,N1,N2,PH1,PH2,HCC1,HCC2,HPFP1,HPFP2,RUPRH1,RUPRH2
0,dekk/mbel_cb_avec_arbr,18.066667,18.800000,64.400000,64.733333,0.155733,0.151467,4.686667,3.233333,8.079813,...,0.360222,0.322000,5.864444,5.806667,19.926366,20.082567,9.113581,9.235677,32.438357,32.540670
1,dekk/mbel_cb_sans_arbr,18.100000,18.700000,64.450000,64.850000,0.155650,0.150900,4.745000,3.255000,8.180380,...,0.353333,0.317000,5.865000,5.815000,19.618432,20.266224,8.976415,9.353365,31.926052,32.738577
2,dekk_cb_avec_arbr,17.492063,18.365079,66.047619,65.984127,0.154603,0.150889,4.369841,3.038095,7.533606,...,0.335820,0.305873,5.841799,5.766667,18.820873,19.060625,8.657769,8.797480,30.489314,30.789434
3,dekk_cb_sans_arbr,17.457831,18.385542,66.060241,65.951807,0.154542,0.150964,4.333735,3.025301,7.471359,...,0.331245,0.303133,5.829317,5.765060,19.107418,19.225314,8.763053,8.867642,31.033094,31.073014
4,dior_cb_avec_arbr,17.647668,18.481865,65.860104,65.678756,0.154756,0.150741,4.507254,3.111399,7.770506,...,0.330086,0.304974,5.838687,5.776166,18.849091,19.341612,8.677742,8.947725,30.514047,31.181659


In [25]:
# Le nom de la variable de sortie a été défini au début du notebook
# output_csv_path = base_dir / "data" / "sols" / "csv" / "processed" / "donnees_typesDeSol.csv"

# Sauvegarder la table de synthèse
df_agrege.to_csv(output_csv_path, index=False, sep=';')

print("✅ Table de synthèse des sols sauvegardée avec succès.")
print(f"   -> Emplacement : {output_csv_path}")

✅ Table de synthèse des sols sauvegardée avec succès.
   -> Emplacement : C:\Users\Cheikhou\Desktop\Ferlo_Sine\maelia-data-diohine-v1\data\sols\csv\processed\donnees_typesDeSol.csv
